# Alignment Format Validation
This report reflects the BAM files Harpy processed to identify obvious formatting issues that may require your attention.

In [ ]:
cat(format(Sys.time(), '🗓️ %d %B, %Y 🕔 %H:%M'))

In [ ]:
library(dplyr)
library(DT)
library(tidyr)
library(scales)

In [ ]:
dt_iframe_height <- function(x){
  # this is exclusively for making interactive plots appear correctly
  # in MyST-rendered html pages
  if (is.data.frame(x)) {
    .x <- nrow(x)
  } else {
    .x <- x
  }
  frameheight <- 200 + (50 * min(10, .x))

  sprintf("
      function(el, x) {
        // Try to resize parent iframe
        if (window.frameElement) {
          window.frameElement.style.width = '100%%';
          window.frameElement.style.height = '%spx';
        }
      }
    ", frameheight)
}

In [ ]:
create_colored_box <- function(value, label, color = "steelblue", width = "200px", height = "90px") {
  .val <- formatC(value, format = "f", big.mark = ",", digits = 2, drop0trailing = T)
  sprintf(
    '<div style="background-color: %s; width: %s; height: %s; display: flex; flex-direction: column; align-items: center; justify-content: center; color: white; border-radius: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.15); padding: 20px; box-sizing: border-box;"><div style="font-size: 14px; font-weight: normal; margin-bottom: 2px; margin-top: 13px; opacity: 0.9; text-transform: uppercase; letter-spacing: 1px;">%s</div><div style="font-size: 48px; font-weight: bold;">%s</div></div>',
    color, width, height, label, .val
    )
}

# Function to display boxes that wrap dynamically
show_boxes <- function(boxes, gap = "15px") {
  box_html <- paste(boxes, collapse = "\n")
  html_content <- sprintf('<div style="display: flex; flex-wrap: wrap; gap: %s; width: 100%%;">%s</div>', gap, box_html)
  IRdisplay::display_html(html_content)
}

In [ ]:
infile <- "test.tsv"
platform <- "haplotagging"

In [ ]:
data <- read.table(infile, header = T)
attention_df <- data %>% select(-1,-2) %>% rowwise() %>% summarise(count = sum(c(nameMismatch, noBX, badBX)))
attention <- sum(attention_df$count > 0)
noMItag <- sum(data$noMI == data$alignments)
noBXtag <- sum(data$noBX == data$alignments)
bxnotlast <- sum(data$bxNotLast > 0)
format_issues <- sum(data$badBX > 0)

In [ ]:
show_boxes(c(
  create_colored_box(nrow(data), "Files", "#aeaeaeff"),
  create_colored_box(attention, "Needs Review", ifelse(attention > 0, "#f6ab3c", "#68ae6bff")),
  create_colored_box(noBXtag, "No BX Tag", ifelse(noBXtag > 0, "#f6ab3c", "#68ae6bff")),
  create_colored_box(noMItag, "No MI Tag", ifelse(noMItag > 0, "#f6ab3c", "#68ae6bff")),
  create_colored_box(noMItag, "Bad Format", ifelse(format_issues > 0, "#f6ab3c", "#68ae6bff")),
  create_colored_box(bxnotlast, "BX Not Last", ifelse(bxnotlast > 0, "#f6ab3c", "#68ae6bff"))
))

## Metrics
Hover over the metric descriptions[^1], how to interpret the data[^2], proper barcode formats[^3], and an explanation of proper alignment records[^4] to better understand what these validation assessments mean.

[^1]:
    | column            | 🟩 pass condition 🟩                                                             | 🟨 fail condition 🟨                                       |
    |:------------------|:---------------------------------------------------------------------------------|:-----------------------------------------------------------|
    | **Name Mismatch** | the file name matches the `@RG ID:` tag in the header                            | file name does not match `@RG ID:` in the header           |
    | **MI tag**        | **any** alignments with `BX:Z` tag also have `MI`                                | **all** reads have `BX:Z` tag present but `MI` tag absent  |
    | **BX:Z tag**      | **any** `BX:Z` tags present                                                      | **all** alignments lack `BX:Z` tag                         |
    | **BX:Z last**     | **all** reads have `BX:Z` as final tag in alignment records                      | **at least 1 read** doesn't have `BX:Z` tag as final tag   |
    | **Format**        | **all** alignments with `BX:Z` tag have properly formatted barcodes for their chemistry | **any** `BX:Z` barcodes have incorrectly formatted barcodes for their chemistry |

[^2]:
    The `harpy validate bam ...` command created a `validate.bam.tsv` file
    that summarizes the results included in this report. This file contains
    a tab-delimited table with the columns: `file`,	`nameMismatch`, `alignments`, `format`, `noBX`, and `badBX`.
    These columns are defined below and the severities of their issues are given as colored emoji where ⬜ is minor,
    🔶 is moderate, 🛑 is critical.

    **file**: The name of the BAM file

    **nameMismatch 🛑**: The sample name of the file inferred from name of the file (i.e. Harpy assumes `sample1.bam` to be `sample1`) does not match the `@RG ID` tag in the alignment file.

    **alignments**: The total number of alignment records in the file

    **noMI 🛑**: Alignment records lack `MI` tag (`MI:i` or `MI:Z`)

    **noBX ⬜**: The number of alignments that do not have a `BX:Z` tag in the record.
    If you expect all or some of your reads should have `BX:Z` tags, then further investigation is necessary

    **badBX 🛑**: The barcode in the `BX:Z` tag does not adhere to the proper `r platform` format

    **bxNotLast 🔶**: The `BX:Z` tag in the alignment record is not the last tag
    Only relevant for LEVIATHAN variant calling and can be ignored if not intending to call
    structural variants with LEVIATHAN,otherwise the `BX:Z` tag must be the last comment in an alignment
    record. `harpy align` will automatically move the `BX:Z` tag to the end of the alignment record.

[^3]:
    A proper linked-read alignment file will contain a `BX:Z` tag with an alignment record
    that features a properly-formatted barcode. The formats are:

    **haplotagging**: `AXXCXXBXXDXX` where `X` is a 0-9 integer
    - example: `A02C56B09D88`
    - invalid: if any position is `00` (e.g. `A91C00B42D57`)

    **stlfr**: `X_X_X` where `X` is any integer
    - example: `9_851_22 
    - invalid: if any position is `0` (e.g. `51_0_492`)

    **tellseq/10x**: `ATCGN` nucleotide letters
    - example: `ATATTTACGGGAC`
    - invalid: if any nucleotide is `N` (e.g. `ATACANGGAT`)

    If a barcode is not in the technology-specific format, then it's likely the input FASTQ used for read mapping is the source of the
    issue. You can check those FASTQ files for errors with `harpy validate fastq`. 

[^4]:
    Below is an example of a proper alignment record for a file named `sample1.bam`.
    Note the tag `RG:Z:sample1`, indicating this alignment is associated with `sample1` and
    matches the file name. Also note the correctly formatted haplotagging barcode `BX:Z:A19C01B86D78`
    and the presence of a `MI:` tag. To reduce horizontal scrolling, the alignment sequence and Phred
    scores have been replaced with `SEQUENCE` and `QUALITY`, respectively.

    ```
    A00814:267:HTMH3DRXX:2:1132:26268:10316 113     contig1 6312    60      4S47M1D86M      =       6312    0       SEQUENCE  QUALITY     NM:i:2  MD:Z:23C23^C86  MC:Z:4S47M1D86M AS:i:121        XS:i:86 RG:Z:sample1   MI:i:4040669   BX:Z:A19C01B86D78
    ```

In [ ]:
datatable(
  data,
  escape = F,
  rownames = F,
  filter = 'top',
  width = '100%',
  extensions = 'Buttons',
  fillContainer = T,
  colnames = c("File", "Reads", "Name Mismatch", "MI tag", "BX:Z tag", "BX:Z last", "Format"),
  options = list(
    scrollY = F,
    deferRender = T,
    scrollX = T,
    dom = 'Brtp',
    buttons = list(list(extend = "csv", filename = "validate.bam"))
  )
) %>%
  formatStyle(
    columns = 3:ncol(data),
    # green for values <= 0, yellow for values > 0
    backgroundColor = styleInterval(0, c('#68ae6bff', '#f6ab3c'))
  ) %>%
  formatStyle(
    columns = 2,
    # yellow for values == 0, green for values > 0
    backgroundColor = styleInterval(0, c('#f6ab3c', '#68ae6bff'))
  ) %>%
  htmlwidgets::onRender(dt_iframe_height(data))